In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files

# --- 1. 🛠️ USER CONFIGURATION ---
# VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV
#  IMPORTANT: CHOOSE YOUR MODEL HERE! (and adjust related parameters if needed)
#  Your teacher might tell you which one to use.
#  Options: "MobileNetV2", "EfficientNetB0", "ResNet50", "InceptionV3", "VGG16"
CHOSEN_MODEL_NAME = "MobileNetV2" # <--- CHANGE THIS LINE AS NEEDED
# VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV

# Model-specific configurations (defaults provided, adjust if your chosen model needs different)
MODEL_CONFIG = {
    "MobileNetV2": {
        "module": applications.mobilenet_v2,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "EfficientNetB0": {
        "module": applications.efficientnet, # Base module for EfficientNet
        "model_func_name": "EfficientNetB0", # Specific function name
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "ResNet50": {
        "module": applications.resnet50,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "InceptionV3": { # Representative of GoogLeNet family
        "module": applications.inception_v3,
        "input_size": (299, 299), # Or (75,75) minimum
        "weights": "imagenet",
    },
    "VGG16": { # Similar depth/style to older models like AlexNet
        "module": applications.vgg16,
        "input_size": (224, 224),
        "weights": "imagenet",
    }
}

if CHOSEN_MODEL_NAME not in MODEL_CONFIG:
    print(f"Error: Model '{CHOSEN_MODEL_NAME}' is not configured. Please choose from {list(MODEL_CONFIG.keys())}")
    exit()

SELECTED_MODEL_DETAILS = MODEL_CONFIG[CHOSEN_MODEL_NAME]
IMG_WIDTH, IMG_HEIGHT = SELECTED_MODEL_DETAILS["input_size"]
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)

# General training parameters
BATCH_SIZE = 32
EPOCHS = 10  # Adjust as needed for your dataset and model (can be 5-10 for quick exam)
LEARNING_RATE = 0.001

# Dataset parameters
DATASET_ZIP_NAME = 'my_image_dataset.zip' # The name of the zip file you will upload
DATASET_EXTRACT_PATH = 'extracted_dataset_content' # Folder where dataset will be extracted

# --- 2. 📂 Dataset Upload and Preparation ---
# (This part is for Google Colab)
from google.colab import files

print(f"Please upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"\n'{DATASET_ZIP_NAME}' uploaded successfully!")
    if os.path.exists(DATASET_EXTRACT_PATH):
        print(f"Cleaning up existing directory: {DATASET_EXTRACT_PATH}")
        import shutil
        shutil.rmtree(DATASET_EXTRACT_PATH) # Remove old extracted content
    os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    # Try to find the actual root directory of your classes
    # (e.g., if zip extracts to 'extracted_dataset_content/my_actual_folder/class_a')
    extracted_items = os.listdir(DATASET_EXTRACT_PATH)
    if not extracted_items:
        print(f"Error: The extracted directory '{DATASET_EXTRACT_PATH}' is empty.")
        exit()

    potential_dataset_root = os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])
    if os.path.isdir(potential_dataset_root) and len(os.listdir(potential_dataset_root)) > 0:
         # Check if this first item is a directory and contains other items (likely class folders)
        sub_items = [d for d in os.listdir(potential_dataset_root) if os.path.isdir(os.path.join(potential_dataset_root, d))]
        if len(sub_items) > 1: # More than one sub-directory, assume it's the root
            dataset_dir = potential_dataset_root
        else: # Otherwise, assume classes are directly under DATASET_EXTRACT_PATH
            dataset_dir = DATASET_EXTRACT_PATH
    else: # Or classes are directly under DATASET_EXTRACT_PATH
        dataset_dir = DATASET_EXTRACT_PATH

    print(f"Using dataset directory: {dataset_dir}")

    if not os.path.exists(dataset_dir) or not any(os.path.isdir(os.path.join(dataset_dir, i)) for i in os.listdir(dataset_dir)):
        print(f"ERROR: Dataset directory '{dataset_dir}' does not contain class subfolders. "
              "Please check your ZIP file structure. "
              "It should contain a root folder (e.g., 'my_image_dataset'), "
              "and inside it, subfolders for each class (e.g., 'dogs', 'cats').")
        exit()
else:
    print(f"ERROR: '{DATASET_ZIP_NAME}' not found. Please upload the correct file.")
    exit()

# --- 3. 🖼️ Load Data with Preprocessing ---
print("\nLoading and preprocessing data...")

# Get the preprocessing function for the chosen model
preprocess_input_fn = getattr(SELECTED_MODEL_DETAILS["module"], "preprocess_input", None)
if preprocess_input_fn is None: # For EfficientNet B0-B7, it's directly in applications module
    if CHOSEN_MODEL_NAME.startswith("EfficientNet"):
         preprocess_input_fn = getattr(applications.efficientnet, "preprocess_input")


def preprocess_dataset_images(image, label):
    if preprocess_input_fn:
        image = preprocess_input_fn(image) # Apply model-specific preprocessing
    return image, label

# Create training dataset (80% of data)
train_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' # For categorical_crossentropy
)

# Create validation dataset (20% of data)
validation_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Apply model-specific preprocessing to the datasets
if preprocess_input_fn:
    print(f"Applying model-specific preprocessing for {CHOSEN_MODEL_NAME}...")
    train_dataset = train_dataset.map(preprocess_dataset_images, num_parallel_calls=tf.data.AUTOTUNE)
    validation_dataset = validation_dataset.map(preprocess_dataset_images, num_parallel_calls=tf.data.AUTOTUNE)
else: # If no specific preprocess_input, use basic rescaling
    print("Applying basic rescaling (pixels to [0,1])...")
    rescale_layer = layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (rescale_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    validation_dataset = validation_dataset.map(lambda x, y: (rescale_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)


# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. 🏗️ Build the Transfer Learning Model ---
print(f"\nBuilding model with base: {CHOSEN_MODEL_NAME}")

# Load the base model
input_shape = IMAGE_SIZE + (3,)
base_model_module = SELECTED_MODEL_DETAILS["module"]

if "model_func_name" in SELECTED_MODEL_DETAILS: # For models like EfficientNetB0
    base_model_constructor = getattr(base_model_module, SELECTED_MODEL_DETAILS["model_func_name"])
else: # For models like MobileNetV2, ResNet50, etc.
    base_model_constructor = getattr(base_model_module, CHOSEN_MODEL_NAME)

base_model = base_model_constructor(
    input_shape=input_shape,
    include_top=False,  # Do not include the model's final classifier layer
    weights=SELECTED_MODEL_DETAILS["weights"]
)

# Freeze the base model (so we only train our new classifier)
base_model.trainable = False

# Create the new model on top
inputs = tf.keras.Input(shape=input_shape)
# If NOT using .map(preprocess_dataset_images), the preprocess_input can be a layer
# x = preprocess_input_fn(inputs) # This is now handled by .map()
x = base_model(inputs, training=False) # Ensure base_model runs in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x) # Regularization
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# --- 5. ⚙️ Compile the Model ---
print("\nCompiling the model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# --- 6. 🚀 Train the Model ---
print("\nStarting model training...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

# --- 7. 📊 Evaluate and Plot ---
print("\nEvaluating model and plotting history...")
val_loss, val_accuracy = model.evaluate(validation_dataset)
print(f"\nFinal Validation Loss: {val_loss:.4f}")
print(f"Final Validation Accuracy: {val_accuracy*100:.2f}%")

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Use len(acc) for actual number of epochs run

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

# --- 8. 🔮 Predict on a New Image ---
def predict_single_new_image(trained_model, class_names_list, img_size_tuple, model_preprocess_fn=None):
    print("\nUpload an image for prediction:")
    uploaded_img_dict = files.upload()

    if not uploaded_img_dict:
        print("No file uploaded.")
        return

    file_path = list(uploaded_img_dict.keys())[0]

    try:
        img = tf.keras.preprocessing.image.load_img(file_path, target_size=img_size_tuple)
        img_array = tf.keras.preprocessing.image.img_to_array(img)

        # Important: Apply the SAME preprocessing as used for training
        if model_preprocess_fn:
            img_array_processed = model_preprocess_fn(img_array)
        else: # Basic rescale if no specific function (e.g. if it was done via layer initially)
            img_array_processed = img_array / 255.0

        img_batch = tf.expand_dims(img_array_processed, 0) # Create a batch

        predictions_output = trained_model.predict(img_batch)

        # Softmax is already applied by the model's last layer
        predicted_index = np.argmax(predictions_output[0])
        confidence = np.max(predictions_output[0]) * 100
        predicted_class_name = class_names_list[predicted_index]

        plt.imshow(img) # Show original uploaded image (before model-specific preprocessing)
        plt.title(f"Predicted: {predicted_class_name} ({confidence:.2f}%)")
        plt.axis("off")
        plt.show()

        print(f"The image is predicted as: '{predicted_class_name}' with {confidence:.2f}% confidence.")

    except Exception as e:
        print(f"Error processing or predicting image: {e}")

# Call prediction function
# Ensure the correct preprocessing function is passed if it wasn't part of the saved model graph
# (here, preprocess_input_fn is the one we used for the dataset)
active_preprocess_fn_for_single_image = None
if preprocess_input_fn: # If we used a specific one for dataset
    active_preprocess_fn_for_single_image = preprocess_input_fn

predict_single_new_image(model, class_names, IMAGE_SIZE, active_preprocess_fn_for_single_image)

print("\n--- End of General Classification Script ---")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files
import shutil # For cleaning up directories

# --- 1. 🛠️ USER CONFIGURATION ---
# VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV
#  IMPORTANT: CHOOSE YOUR MODEL HERE!
#  Your teacher might tell you which one to use.
#
#  Popular Options (around 10 provided):
#  "MobileNetV2", "EfficientNetB0", "ResNet50V2", "InceptionV3", "VGG16",
#  "DenseNet121", "Xception", "InceptionResNetV2", "MobileNetV3Large", "EfficientNetB1"
#
CHOSEN_MODEL_NAME = "MobileNetV2"  # <--- CHANGE THIS LINE AS NEEDED
# VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV

# Model-specific configurations
MODEL_CONFIG = {
    "MobileNetV2": {
        "module": applications.mobilenet_v2,
        "model_func_itself": applications.MobileNetV2, # Direct model function
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "EfficientNetB0": {
        "module": applications.efficientnet,
        "model_func_itself": applications.EfficientNetB0,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "EfficientNetB1": { # Slightly larger EfficientNet
        "module": applications.efficientnet,
        "model_func_itself": applications.EfficientNetB1,
        "input_size": (240, 240), # B1 default
        "weights": "imagenet",
    },
    "ResNet50V2": { # V2 uses improved residual blocks
        "module": applications.resnet_v2, # Note: module is resnet_v2
        "model_func_itself": applications.ResNet50V2,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "InceptionV3": { # Representative of GoogLeNet family
        "module": applications.inception_v3,
        "model_func_itself": applications.InceptionV3,
        "input_size": (299, 299), # Or (75,75) minimum
        "weights": "imagenet",
    },
    "VGG16": { # Classic deep model
        "module": applications.vgg16,
        "model_func_itself": applications.VGG16,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "DenseNet121": { # Densely Connected Convolutional Networks
        "module": applications.densenet,
        "model_func_itself": applications.DenseNet121,
        "input_size": (224, 224),
        "weights": "imagenet",
    },
    "Xception": { # "Extreme Inception"
        "module": applications.xception,
        "model_func_itself": applications.Xception,
        "input_size": (299, 299), # Or (71,71) minimum
        "weights": "imagenet",
    },
    "InceptionResNetV2": { # Combines Inception with Residual connections
        "module": applications.inception_resnet_v2,
        "model_func_itself": applications.InceptionResNetV2,
        "input_size": (299, 299), # Or (75,75) minimum
        "weights": "imagenet",
    },
    "MobileNetV3Large": { # Newer MobileNet
        "module": applications.mobilenet_v3,
        "model_func_itself": applications.MobileNetV3Large,
        "input_size": (224, 224),
        "weights": "imagenet",
    }
    # You can add more models here following the same pattern
}

if CHOSEN_MODEL_NAME not in MODEL_CONFIG:
    print(f"Error: Model '{CHOSEN_MODEL_NAME}' is not configured. Please choose from {list(MODEL_CONFIG.keys())}")
    exit()

SELECTED_MODEL_DETAILS = MODEL_CONFIG[CHOSEN_MODEL_NAME]
IMG_WIDTH, IMG_HEIGHT = SELECTED_MODEL_DETAILS["input_size"]
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)

# General training parameters
BATCH_SIZE = 32
EPOCHS = 10  # Adjust as needed (5-10 for quick exam, more for better results)
LEARNING_RATE = 0.001

# Dataset parameters
DATASET_ZIP_NAME = 'my_image_dataset.zip'
DATASET_EXTRACT_PATH = 'extracted_dataset_content'

# --- 2. 📂 Dataset Upload and Preparation ---
from google.colab import files

print(f"Please upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"\n'{DATASET_ZIP_NAME}' uploaded successfully!")
    if os.path.exists(DATASET_EXTRACT_PATH):
        print(f"Cleaning up existing directory: {DATASET_EXTRACT_PATH}")
        shutil.rmtree(DATASET_EXTRACT_PATH)
    os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    extracted_items = os.listdir(DATASET_EXTRACT_PATH)
    if not extracted_items:
        print(f"Error: The extracted directory '{DATASET_EXTRACT_PATH}' is empty.")
        exit()

    # Try to find the actual root directory of your classes
    # This logic assumes the first item in the extracted path is your dataset's root folder, OR
    # the class folders are directly under DATASET_EXTRACT_PATH.
    dataset_dir_found = False
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])):
        # Likely a single root folder was zipped
        dataset_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])
        if any(os.path.isdir(os.path.join(dataset_dir, i)) for i in os.listdir(dataset_dir)):
             dataset_dir_found = True

    if not dataset_dir_found:
        # Assume class folders are directly under DATASET_EXTRACT_PATH
        if any(os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, i)) for i in extracted_items):
            dataset_dir = DATASET_EXTRACT_PATH
            dataset_dir_found = True

    if not dataset_dir_found:
        print(f"ERROR: Could not automatically determine the dataset directory with class subfolders "
              f"within '{DATASET_EXTRACT_PATH}'. Please check your ZIP file structure. "
              "It should typically contain a single root folder (e.g., 'my_image_dataset'), "
              "and inside that, subfolders for each class (e.g., 'dogs', 'cats').")
        exit()

    print(f"Using dataset directory: {dataset_dir}")

    if not any(os.path.isdir(os.path.join(dataset_dir, i)) for i in os.listdir(dataset_dir)):
        print(f"ERROR: No class subfolders found in '{dataset_dir}'.")
        exit()

else:
    print(f"ERROR: '{DATASET_ZIP_NAME}' not found. Please upload the correct file.")
    exit()

# --- 3. 🖼️ Load Data with Preprocessing ---
print("\nLoading and preprocessing data...")

preprocess_input_fn = getattr(SELECTED_MODEL_DETAILS["module"], "preprocess_input", None)

def preprocess_dataset_images(image, label):
    # image is already resized by image_dataset_from_directory
    if preprocess_input_fn:
        image = preprocess_input_fn(image)
    else: # Fallback if a model's module doesn't have preprocess_input (rare for Keras Apps)
        image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

validation_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names}")
print(f"Number of classes: {num_classes}")

print(f"Applying model-specific preprocessing for {CHOSEN_MODEL_NAME}...")
train_dataset = train_dataset.map(preprocess_dataset_images, num_parallel_calls=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(preprocess_dataset_images, num_parallel_calls=tf.data.AUTOTUNE)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. 🏗️ Build the Transfer Learning Model ---
print(f"\nBuilding model with base: {CHOSEN_MODEL_NAME}")

input_shape_tuple = IMAGE_SIZE + (3,)
base_model_constructor = SELECTED_MODEL_DETAILS["model_func_itself"]

base_model = base_model_constructor(
    input_shape=input_shape_tuple,
    include_top=False,
    weights=SELECTED_MODEL_DETAILS["weights"]
)
base_model.trainable = False # Freeze the base

# Create the new model on top
inputs = tf.keras.Input(shape=input_shape_tuple)
# The preprocessing is now handled by the .map() on the dataset.
# So, the input to the model is already preprocessed.
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
x = layers.BatchNormalization()(x) # Adding Batch Norm can help stabilize training
x = layers.Dropout(0.3, name="top_dropout")(x) # Regularization
outputs = layers.Dense(num_classes, activation='softmax', name="predictions")(x)

model = models.Model(inputs, outputs)

# --- 5. ⚙️ Compile the Model ---
print("\nCompiling the model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# --- 6. 🚀 Train the Model ---
print("\nStarting model training...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

# --- 7. 📊 Evaluate and Plot ---
print("\nEvaluating model and plotting history...")
val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=0) # verbose=0 to keep output clean
print(f"\nFinal Validation Loss: {val_loss:.4f}")
print(f"Final Validation Accuracy: {val_accuracy*100:.2f}%")

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss_hist = history.history['loss'] # Renamed to avoid conflict with 'loss' from evaluate
val_loss_hist = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title(f'{CHOSEN_MODEL_NAME} - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss_hist, label='Training Loss')
plt.plot(epochs_range, val_loss_hist, label='Validation Loss')
plt.legend(loc='upper right')
plt.title(f'{CHOSEN_MODEL_NAME} - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

# --- 8. 🔮 Predict on a New Image ---
def predict_single_new_image(trained_model, class_names_list, img_size_tuple, model_specific_preprocess_fn):
    print("\nUpload an image for prediction:")
    uploaded_img_dict = files.upload()

    if not uploaded_img_dict:
        print("No file uploaded.")
        return

    file_path = list(uploaded_img_dict.keys())[0]

    try:
        img = tf.keras.preprocessing.image.load_img(file_path, target_size=img_size_tuple)
        img_array_original = tf.keras.preprocessing.image.img_to_array(img) # For display & preprocessing

        # Apply the SAME preprocessing as used for training data
        if model_specific_preprocess_fn:
            img_array_processed = model_specific_preprocess_fn(img_array_original)
        else: # Fallback to simple rescale if no specific function provided
            img_array_processed = img_array_original / 255.0

        img_batch = tf.expand_dims(img_array_processed, 0)

        predictions_output = trained_model.predict(img_batch)

        predicted_index = np.argmax(predictions_output[0])
        confidence = np.max(predictions_output[0]) * 100
        predicted_class_name = class_names_list[predicted_index]

        plt.imshow(img) # Show original uploaded image
        plt.title(f"Predicted: {predicted_class_name} ({confidence:.2f}%)")
        plt.axis("off")
        plt.show()

        print(f"The image is predicted as: '{predicted_class_name}' with {confidence:.2f}% confidence.")

    except Exception as e:
        print(f"Error processing or predicting image: {e}")
        print(f"Make sure the uploaded file is a valid image format (JPG, PNG etc.).")

predict_single_new_image(model, class_names, IMAGE_SIZE, preprocess_input_fn)

print("\n--- End of General Classification Script ---")